In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
import numpy as np
import pandas as pd
from pathlib import Path
from utils.benchmarks import BENCHMARKS_TO_PATHS


In [ ]:
%%RecordEvent
%%time
### cell 0 ###

benchmark_name = "nyc-flight"
factor = 2
flights_df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "nyc_flights.csv"
)
flights_df = pd.concat([flights_df] * factor, ignore_index=True)

In [ ]:
%%RecordEvent
%%time
### cell 1 ###

flights_df.shape, flights_df.columns, flights_df.dtypes

In [ ]:
%%RecordEvent
%%time
### cell 2 ###

flights_df.dest.unique()
flights_df.head(10)

In [ ]:
%%RecordEvent
%%time
### cell 3 ###

flights_df["dest"][flights_df["dest"] == "SEA"].value_counts()

In [ ]:
%%RecordEvent
%%time
### cell 4 ###

flights_df["carrier"][flights_df["dest"] == "SEA"].value_counts()

In [ ]:
%%RecordEvent
%%time
### cell 5 ###

len(flights_df["tailnum"][flights_df["dest"] == "SEA"].unique())

In [ ]:
%%RecordEvent
%%time
### cell 6 ###

flights_df["arr_delay"][flights_df["dest"] == "SEA"].mean()

In [ ]:
%%RecordEvent
%%time
### cell 7 ###

f = flights_df[flights_df["dest"] == "SEA"].groupby("origin").size()
f_total = len(flights_df[flights_df.dest == "SEA"])
f.loc["EWR"] / f_total, f.loc["JFK"] / f_total

In [ ]:
%%RecordEvent
%%time
### cell 8 ###

df = flights_df.groupby(["month", "day"], as_index=False).agg({"dep_delay": np.mean})
df2 = flights_df.groupby(["month", "day"], as_index=False).agg({"arr_delay": np.mean})
df.loc[df["dep_delay"].idxmax()], df2.loc[df2["arr_delay"].idxmax()]

In [ ]:
%%RecordEvent
%%time
### cell 9 ###

df

In [ ]:
%%RecordEvent
%%time
### cell 10 ###

df = flights_df.groupby(["day", "month"], as_index=False).agg(
    {"arr_delay": np.mean, "dep_delay": np.mean}
)
df["total_delay"] = df["arr_delay"] + df["dep_delay"]
df.sort_values("total_delay", ascending=False).head(1)

In [ ]:
%%RecordEvent
%%time
### cell 11 ###

ds = flights_df.dropna(subset=["dep_delay"]).groupby(["month"])["dep_delay"].mean()
ds

In [ ]:
%%RecordEvent
%%time
### cell 12 ###

dt = flights_df.dropna(subset=["dep_delay"]).groupby(["hour"])["dep_delay"].mean()
dt

In [ ]:
%%RecordEvent
%%time
### cell 13 ###

df = flights_df
df["speed"] = df["distance"] / df["air_time"]
df[df["speed"] == df.speed.max()]

In [ ]:
%%RecordEvent
%%time
### cell 14 ###

count = len(flights_df)
df = flights_df.groupby(["carrier", "flight", "dest"]).size().reset_index(name="Size")
for i in df.index:
    if df.loc[i]["Size"] == 365:
        print(
            "Carrier: %s, Flight: %s, Destination: %s"
            % (df.loc[i]["carrier"], df.loc[i]["flight"], df.loc[i]["dest"])
        )

In [ ]:
%%RecordEvent
%%time
### cell 15 ###

gdf = flights_df
counts = gdf.groupby(["carrier", "flight", "dest"]).size().reset_index(name="size")
complete_year = counts[counts["size"] == 365]
complete_year[["carrier", "flight", "dest"]]

In [ ]:
%%RecordEvent
%%time
### cell 16 ###

df = flights_df[flights_df["month"] == 6]
df = df.groupby("carrier", as_index=False).agg(
    {"arr_delay": np.mean, "dep_delay": np.mean}
)
df["total_delay"] = df["arr_delay"] + df["dep_delay"]
for i in df.index:
    if df.loc[i]["total_delay"] == df["total_delay"].min():
        print(df.loc[i]["carrier"], df.loc[i]["total_delay"])
df

In [ ]:
%%RecordEvent
%%time
### cell 17 ###

weather_df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "nyc_weather.csv"
)
df = flights_df
df_c = pd.merge(df, weather_df, on=["month", "day", "hour", "origin"])
df_c.head(10)